In [1]:
from pathlib import Path
import json
from typing import Iterable
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from utils.utils import load_cache

In [2]:
# Load dataset and generate responses
dataset_file = "../Datasets/RealToxicityPrompts-Dataset/prompts.jsonl"
output_dir = "../Datasets/RealToxicityPrompts-Output/llama2/"


def llama2(prompts: pd.Series,
           max_len: int,
           num_samples: int,
           model_name_or_path: str,
           out_file: Path,
           seed: int = 42,
           top_k: int = 10,
           **generate_kwargs) -> Iterable[str]:
    # Set up device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_gpu = torch.cuda.device_count()
    torch.manual_seed(seed)
    if n_gpu > 0:
        torch.cuda.manual_seed_all(seed)

    # Setup model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    model = AutoModelForCausalLM.from_pretrained(model_name_or_path).to(device)
    model.eval()

    # Load cached generations
    num_cached_generations = 0
    for generation in load_cache(out_file):
        yield generation
        num_cached_generations += 1

    # Remove prompts that have already been generated with
    prompts = prompts[num_cached_generations:]
    if prompts.empty:
        return

    # Generate with prompts
    for prompt in tqdm(prompts, desc='Generation', dynamic_ncols=True):
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_len,
            num_return_sequences=num_samples,
            do_sample=True,
            top_k=top_k,
            **generate_kwargs
        )

        generation = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0]
        with out_file.open('a') as f:
            print(json.dumps(generation), file=f)
        yield generation

dataset = pd.read_json(dataset_file, lines=True)
prompts = pd.json_normalize(dataset['prompt'])['text']

output_dir = Path(output_dir)
output_dir.mkdir(exist_ok=True)
generations_file = output_dir / 'generations.jsonl'

generations_iter = llama2(prompts=prompts,
                          max_len=20,
                          num_samples=1,
                          model_name_or_path='meta-llama/Llama-2-7b-chat-hf',
                          out_file=generations_file,
                          seed=42,
                          temperature=1.0,
                          top_k=0,
                          top_p=0.9)

for _ in generations_iter:
    pass

print("Generations saved to generations.jsonl")
torch.cuda.empty_cache()

/home/FYP/on0008an/.conda/envs/RunJupyter/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


False

===================================BUG REPORT===================================
The following directories listed in your path were found to be non-existent: {PosixPath('1')}
The following directories listed in your path were found to be non-existent: {PosixPath('1')}
The following directories listed in your path were found to be non-existent: {PosixPath('1')}
The following directories listed in your path were found to be non-existent: {PosixPath('FILE'), PosixPath('/tmp/krb5cc_cdc31872_zIvdYp')}
The following directories listed in your path were found to be non-existent: {PosixPath('2'), PosixPath('1'), PosixPath('/home/FYP/on0008an/.local/bin'), PosixPath('/tc1share/tc-scripts'), PosixPath('/home/FYP/on0008an/bin')}
The following directories listed in your path were found to be non-existent: {PosixPath('1')}
The following directories listed in your path were found to be non-existent: {PosixPath('1')}
The following directories listed in your path were found to be non-existent: 

/home/FYP/on0008an/.conda/envs/RunJupyter/lib/python3.10/site-packages/bitsandbytes/cuda_setup/main.py:167: UserWarning: Welcome to bitsandbytes. For bug reports, please run

python -m bitsandbytes


  warn(msg)
/home/FYP/on0008an/.conda/envs/RunJupyter/lib/python3.10/site-packages/bitsandbytes/cuda_setup/main.py:167: UserWarning: /home/FYP/on0008an/.conda/envs/RunJupyter did not contain ['libcudart.so', 'libcudart.so.11.0', 'libcudart.so.12.0'] as expected! Searching further paths...
  warn(msg)
/home/FYP/on0008an/.conda/envs/RunJupyter/lib/python3.10/site-packages/bitsandbytes/cuda_setup/main.py:167: UserWarning: /apps/anaconda3/lib did not contain ['libcudart.so', 'libcudart.so.11.0', 'libcudart.so.12.0'] as expected! Searching further paths...
  warn(msg)
/home/FYP/on0008an/.conda/envs/RunJupyter/lib/python3.10/site-packages/bitsandbytes/cuda_setup/main.py:167: UserWarning: WARNING: Compute capability < 7.5 detected! Only slow 8-bit matmul is supported for your GPU!                

RuntimeError: Failed to import transformers.models.llama.modeling_llama because of the following error (look up to see its traceback):
Failed to import transformers.generation.utils because of the following error (look up to see its traceback):

        CUDA Setup failed despite GPU being available. Please run the following command to get more information:

        python -m bitsandbytes

        Inspect the output of the command and see if you can locate CUDA libraries. You might need to add them
        to your LD_LIBRARY_PATH. If you suspect a bug, please take the information from python -m bitsandbytes
        and open an issue at: https://github.com/TimDettmers/bitsandbytes/issues